In [2]:
%cd ..

/home/blanka/Multi-Domain-Pruning


## Load & print sample

In [16]:
import pickle
import os
import pandas as pd
from IPython.display import display
import glob

sample_label_path = "/data/blanka/DATASETS/SPN/YOLOv8x/label"
sample_data_path = "/data/blanka/DATASETS/SPN/YOLOv8x/data"

sample_pattern = "11_*.pkl"

# Find matching label file
label_files = glob.glob(os.path.join(sample_label_path, sample_pattern))
if not label_files:
    raise FileNotFoundError(f"No label files matching pattern {sample_pattern}")
label_file = label_files[0]  # Take the first match

# Find matching data file
data_files = glob.glob(os.path.join(sample_data_path, sample_pattern))
if not data_files:
    raise FileNotFoundError(f"No data files matching pattern {sample_pattern}")
data_file = data_files[0]

print(data_file)

label_df = pd.read_pickle(label_file)
print(label_df.to_string())

state_df = pd.read_pickle(data_file)
print(state_df.to_string())

/data/blanka/DATASETS/SPN/YOLOv8x/data/11_1_11.pkl
    recall  precision   map50   map90  n_params  recall_init  precision_init  map50_init  map90_init  n_params_init  n_layer_channels
11  0.8684     0.7173  0.7784  0.5854     68.07       0.8731           0.842      0.8749      0.6869          68.13             320.0
    alpha  is_pruned   in_ch  out_ch  kernel  stride  pad  n_pruned_ch  prev_n_params  prev_map50
0     0.0        1.0     3.0    80.0     3.0     2.0  1.0          0.0          68.13      0.8749
1     0.0        1.0    80.0   160.0     3.0     2.0  1.0          0.0          68.13      0.8749
2     0.0        1.0   160.0    80.0     1.0     1.0  0.0          0.0          68.13      0.8749
3     0.0        1.0   160.0    68.0     1.0     1.0  0.0          0.0          68.13      0.8749
4     0.0        1.0   352.0   160.0     1.0     1.0  0.0          0.0          68.13      0.8749
5     0.0        1.0    68.0    80.0     3.0     1.0  1.0          0.0          68.13      0.

## Encode & decode state/label

In [22]:
from state_predictor.coder import Coder

coder = Coder(state_df, label_df, alpha_range=(0, 2.2))
encoded_state = coder.encode_state(state_df)
encoded_label = coder.encode_label(label_df)

## Normalize & denormalize label

In [23]:
from utils.common_utils import normalize, denormalize

dmap = -0.2
range = (0, 1)

norm_dmap = normalize(dmap, range)
print(norm_dmap)
denorm_dmap = denormalize(norm_dmap, range)
print(denorm_dmap)

-1.4
-0.19999999999999996


## Test SPN model (predict)

In [3]:
from utils.config_parser import ConfigParser
from src.model.spn_handler import SPNHandler

# Read and save config file
conf = ConfigParser.read("/data/blanka/runs/SPN/YOLOv8x/prev_features_only/20250523_074022_1705bc_optuna/settings.ini")

# load or define SPN model
spn_handler = SPNHandler(conf, run_name="20250523_074022_1705bc_optuna")
spn_handler.create(is_pretrained=True)
pred_spars, pred_dmap = spn_handler.predict(encoded_state.unsqueeze(dim=0))

# Denormalize the encoded label
decoded_spars, decoded_dmap = denormalize(encoded_label, value_range=(0, 1))

# Calculate spacing based on the longest label
spacing = max(len(f"{decoded_dmap:.4f}"), len(f"{pred_dmap:.4f}"), len(f"{decoded_spars:.4f}"), len(f"{pred_spars:.4f}"))

# Print the values in the specified format with equal spacing
print(f"dmap:\n  gt:   {decoded_dmap:>{spacing}.4f}\n  pred: {pred_dmap:>{spacing}.4f}")
print(f"\nspars:\n  gt:   {decoded_spars:>{spacing}.4f}\n  pred: {pred_spars:>{spacing}.4f}")



/home/blanka/Multi-Domain-Pruning/src/model/spn_handler.py:86: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_location="cpu")


NameError: name 'encoded_state' is not defined